# Chapter 04 — Self-Attention from Scratch

Attention is the mechanism that transformed natural language processing forever. Before attention, models processed sequences left-to-right with a fixed context window. Attention allows every position in a sequence to directly 'look at' every other position, enabling the model to capture long-range dependencies that were previously impossible.

### Glossary / Glossário

• self-attention — mechanism where each token attends to all other tokens in the sequence. / Mecanismo onde cada token atende a todos os outros tokens na sequência.
• Q/K/V — Query, Key, Value: the three learned projections in attention. Query asks "what am I looking for?", Key answers "what do I contain?", Value provides the information. / As três projeções aprendidas na atenção. Query pergunta "o que estou procurando?", Key responde "o que eu contém?", Value fornece a informação.
• softmax — function that converts raw scores into a probability distribution summing to 1. / Função que converte scores brutos em distribuição de probabilidade que soma 1.
• causal mask — triangular mask preventing tokens from attending to future positions. / Máscara triangular que impede tokens de atender a posições futuras.
• multi-head attention — running multiple attention operations in parallel, each learning different patterns. / Múltiplas operações de atenção em paralelo, cada uma aprendendo padrões diferentes.
• RoPE — Rotary Position Embedding, encodes token positions via rotation matrices applied to Q and K. / Rotary Position Embedding, codifica posições dos tokens via matrizes de rotação aplicadas a Q e K.

In [ ]:
import torch
import torch.nn.functional as F
import math

def self_attention(x, W_q, W_k, W_v, mask=None):
    """
    x: (batch, seq_len, d_model)
    W_q, W_k, W_v: (d_model, d_k)
    """
    Q = x @ W_q  # Query projection — "what am I looking for?"
    K = x @ W_k  # Key projection — "what do I contain?"
    V = x @ W_v  # Value projection — "what information do I return if matched?"

    d_k = Q.size(-1)  # key dimension — used for scaling to prevent gradient vanishing
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # raw attention scores — how much each token "wants" each other token

    # Causal mask: prevent attending to future tokens
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))  # causal mask — prevents tokens from seeing the future

    attn_weights = F.softmax(scores, dim=-1)  # normalized attention — probabilities summing to 1 per row
    output = attn_weights @ V  # (batch, seq_len, d_k)
    return output, attn_weights

# === Real sentence demo ===
sentence = "O gato sentou no tapete porque ele estava cansado"
words = sentence.split()  # 9 tokens

torch.manual_seed(42)
d_model = 16  # embedding dimension (small for readability)
vocab = {}  # word -> index mapping
for w in words:
    if w not in vocab:
        vocab[w] = len(vocab)
embeddings = torch.randn(len(vocab), d_model)  # random word embeddings (untrained)

# Build input: look up embedding for each word
x = torch.stack([embeddings[vocab[w]] for w in words]).unsqueeze(0)  # (1, 9, 16)

# Random Q/K/V projections (untrained)
W_q = torch.randn(d_model, d_model)  # Query projection weights
W_k = torch.randn(d_model, d_model)  # Key projection weights
W_v = torch.randn(d_model, d_model)  # Value projection weights

_, attn_weights = self_attention(x, W_q, W_k, W_v)

# Print attention matrix with word labels
print("=== Attention Weights (random/untrained) ===")
print(f"{'':>10s}", " ".join(f"{w:>9s}" for w in words))
for i, row_word in enumerate(words):
    row = attn_weights[0, i].detach().numpy()
    print(f"{row_word:>10s}", " ".join(f"{v:>9.3f}" for v in row))

# === Aha: what TRAINED attention looks like ===
# Simulate training: "ele" embedding becomes close to "gato" (coreference learned)
# Near-identity W_q/W_k so embedding similarity drives attention
ele_idx = words.index("ele")
gato_idx = words.index("gato")

x_trained = x.clone()
x_trained[0, ele_idx] = x_trained[0, gato_idx] * 0.95 + torch.randn(d_model) * 0.05  # "ele" ≈ "gato"

W_q_trained = torch.eye(d_model) + torch.randn(d_model, d_model) * 0.1  # near-identity query projection
W_k_trained = torch.eye(d_model) + torch.randn(d_model, d_model) * 0.1  # near-identity key projection

_, attn_trained = self_attention(x_trained, W_q_trained, W_k_trained, W_v)

print("\n=== Attention Weights (simulating trained model) ===")
print(f"{'':>10s}", " ".join(f"{w:>9s}" for w in words))
for i, row_word in enumerate(words):
    row = attn_trained[0, i].detach().numpy()
    marker = " <-- looks at 'gato'!" if row_word == "ele" else ""
    print(f"{row_word:>10s}", " ".join(f"{v:>9.3f}" for v in row), marker)

# === Softmax step-by-step for "ele" ===
Q = x_trained @ W_q_trained
K = x_trained @ W_k_trained
scores_ele = (Q[0, ele_idx] @ K[0].T) / math.sqrt(d_model)
probs_ele = F.softmax(scores_ele, dim=-1)

print(f"\n=== Softmax for 'ele' (step by step) ===")
print(f"{'Word':>10s} {'Raw Score':>10s} {'Softmax':>10s}")
for i, w in enumerate(words):
    print(f"{w:>10s} {scores_ele[i].item():>10.3f} {probs_ele[i].item():>10.3f}")
print(f"\nSum of softmax = {probs_ele.sum().item():.4f} (always 1.0)")

---

**Course:** Aprenda LLM | **Chapter 04**

This notebook is part of the [Aprenda LLM](https://magestic.dev) course.